# Data Preprocessing
Load raw data, clean, merge from multiple sources

In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

RAW_DATA_DIR = '../data/raw'
PROCESSED_DATA_DIR = '../data/processed'

print('Starting data preprocessing')

Starting data preprocessing


## Load Raw Data

In [2]:
# Load all CSV files from raw data directory
raw_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith('.csv')]

dfs = {}
for file in raw_files:
    file_path = os.path.join(RAW_DATA_DIR, file)
    df = pd.read_csv(file_path)
    dfs[file.replace('.csv', '')] = df

for name, df in dfs.items():
    print(f'\n{name}:')
    print(df.head())


cbsl_inflation_rate:
   Year  Inflation_Rate
0  2020             4.6
1  2021             4.3
2  2022             6.7
3  2023             6.3
4  2024             2.2

cbsl_policy_rate:
   Year  Policy_Rate
0  2023         8.50
1  2024         8.00
2  2025         7.75
3  2026         7.75

worldbank_current_account_percent_gdp:
   Year  Current_Account_Percent_GDP
0  2005                    -2.663822
1  2006                    -5.299428
2  2007                    -4.330416
3  2008                    -9.543195
4  2009                    -0.510386

worldbank_gdp_growth:
   Year  GDP_Growth
0  2005    6.241748
1  2006    7.668292
2  2007    6.796826
3  2008    5.950088
4  2009    3.538912

worldbank_government_debt_percent_gdp:
   Year  Government_Debt_Percent_GDP
0  2005                    90.604913
1  2006                    88.699484
2  2007                    84.994417
3  2009                    86.063492
4  2010                    69.253025

worldbank_government_spending_percent_gdp:

## Standardize Dates

In [3]:
# Standardize date columns
for name, df in dfs.items():
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
    elif 'Year' in df.columns:
        df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    
    dfs[name] = df.sort_values('Date')

## Merge Data Sources

In [4]:
# Start with first dataframe and merge others
merged_df = None

for name, df in dfs.items():
    df_temp = df.copy()
    # Drop Year column if exists
    if 'Year' in df_temp.columns:
        df_temp = df_temp.drop('Year', axis=1)
    
    if merged_df is None:
        merged_df = df_temp
    else:
        merged_df = pd.merge(merged_df, df_temp, on='Date', how='outer')

merged_df = merged_df.sort_values('Date').reset_index(drop=True)
merged_df.head()

,Inflation_Rate,Date,Policy_Rate,Current_Account_Percent_GDP,GDP_Growth,Government_Debt_Percent_GDP,Government_Spending_Percent_GDP,Trade_Balance
0,NaN,2005-01-01,NaN,-2.663822,6.241748,90.604913,20.195965,-2.179493e+09
1,NaN,2006-01-01,NaN,-5.299428,7.668292,88.699484,21.147726,-3.110581e+09
2,NaN,2007-01-01,NaN,-4.330416,6.796826,84.994417,20.046704,-3.356826e+09
3,NaN,2008-01-01,NaN,-9.543195,5.950088,NaN,19.211791,-5.572123e+09
4,NaN,2009-01-01,NaN,-0.510386,3.538912,86.063492,20.958420,-2.731111e+09


## Handle Missing Values

In [5]:
print(f'Missing values before:\n{merged_df.isnull().sum()}')

# Forward fill then backward fill
merged_df = merged_df.fillna(method='ffill').fillna(method='bfill')

print(f'\nMissing values after:\n{merged_df.isnull().sum()}')
print(f'\nDataset shape: {merged_df.shape}')

Missing values before:
Inflation_Rate                     15
Date                                0
Policy_Rate                        18
Current_Account_Percent_GDP         2
GDP_Growth                          2
Government_Debt_Percent_GDP        12
Government_Spending_Percent_GDP     3
Trade_Balance                       7
dtype: int64

Missing values after:
Inflation_Rate                     0
Date                               0
Policy_Rate                        0
Current_Account_Percent_GDP        0
GDP_Growth                         0
Government_Debt_Percent_GDP        0
Government_Spending_Percent_GDP    0
Trade_Balance                      0
dtype: int64

Dataset shape: (22, 8)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_31676\4188154724.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_df = merged_df.fillna(method='ffill').fillna(method='bfill')


## Save Preprocessed Data

In [6]:
output_path = os.path.join(PROCESSED_DATA_DIR, 'inflation_master.csv')
merged_df.to_csv(output_path, index=False)

print(f'\nFinal dataset:')
print(merged_df.info())
print(f'\nFirst few rows:')
print(merged_df.head())


Final dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Inflation_Rate                   22 non-null     float64       
 1   Date                             22 non-null     datetime64[ns]
 2   Policy_Rate                      22 non-null     float64       
 3   Current_Account_Percent_GDP      22 non-null     float64       
 4   GDP_Growth                       22 non-null     float64       
 5   Government_Debt_Percent_GDP      22 non-null     float64       
 6   Government_Spending_Percent_GDP  22 non-null     float64       
 7   Trade_Balance                    22 non-null     float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 1.5 KB
None

First few rows:
   Inflation_Rate       Date  Policy_Rate  Current_Account_Percent_GDP  \
0             4.6 2005-01-01         